# 02 · Cleaning and standardisation

Raw prices are not comparable as they come. A ₹40 biscuit pack of 69 g and a ₹60 pack of 400 g tell you nothing until both are turned into **₹ per kg**. This notebook shows each rule I apply and what it does to the real data.

Rules live in `src/normalize.py` (lookup tables + regexes) and are applied by `src/transform.py`. Nothing is dropped at this stage — every raw row becomes one standardised record.

In [1]:
import sys, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 45)

import transform as T
import normalize as N

records = []
for f in sorted((ROOT / "data" / "raw").rglob("*.jsonl")):
    recs, cpi_rows, issues = T.parse_capture(f)
    records.extend(recs)
df = pd.DataFrame(records)
print(f"{len(df)} standardised price records")
df.groupby(["source_code", "price_type"]).size().rename("records")

677 standardised price records


source_code  price_type
DOCA_PMD     retail        252
             wholesale      23
OPEN_PRICES  retail        402
Name: records, dtype: int64

## 1 · Pack size, weight and volume → one canonical unit

Everything is converted to **kg**, **litre** or **unit** (pieces). The normalised price is simply `price ÷ quantity in that unit`.

DoCA is a good test because its reporting unit changes by commodity: most are per kg, but butter is per 100 g, coriander and garlic per 250 g, turmeric per 50 g and eggs per dozen.

In [2]:
doca_now = df[(df.source_code == "DOCA_PMD") & (df.lookback_label == "Price As On")].copy()
doca_now["normalised_price"] = (doca_now["price"] / doca_now["qty_std"]).round(2)
units = doca_now.drop_duplicates("pack_unit_raw")[
    ["product_name_std", "pack_unit_raw", "price", "qty_std", "std_unit", "normalised_price"]]
units.rename(columns={"product_name_std": "commodity", "pack_unit_raw": "reported unit",
                      "price": "reported price (INR)", "qty_std": "quantity", "std_unit": "unit"})

,commodity,reported unit,reported price (INR),quantity,unit,normalised_price
0,Broken Rice,1 Kg,33.84,1.00,kg,33.84
66,Milk,1 Ltr,61.33,1.00,l,61.33
162,Ragi,1 kg,55.85,1.00,kg,55.85
186,Butter (Pasteurised),100 gm,60.55,0.10,kg,605.50
192,Eggs,1 Dozen,83.73,12.00,unit,6.98
210,Coriander (Whole),250 gm,43.83,0.25,kg,175.32
228,Turmeric (Powder),50 gm,16.76,0.05,kg,335.20


For Open Prices the pack size comes from the structured `product_quantity` field when it exists. If not, I try the free-text quantity and then the product name (e.g. *"Paper boat lychee 250ml pet"*). Those parsed sizes are flagged so they can be reviewed. When nothing works, the normalised price stays empty rather than guessed.

In [3]:
op = df[df.source_code == "OPEN_PRICES"]
op["pack_size_source"].value_counts().rename("records")

pack_size_source
api_product_quantity    316
missing                  78
parsed_product_name       5
parsed_quantity           2
price_per_field           1
Name: records, dtype: int64

In [4]:
op[op.pack_size_source.str.startswith("parsed")][
    ["product_name_raw", "quantity_text_raw", "qty_std", "std_unit", "pack_size_source"]]

,product_name_raw,quantity_text_raw,qty_std,std_unit,pack_size_source
293,Brit 50-50 maska chaska 40g (20),NaN,0.04,kg,parsed_product_name
454,Kinley Mineral Water 1ltr,NaN,1.00,l,parsed_product_name
477,Diet Coke Can 250ml,NaN,0.25,l,parsed_product_name
567,7up Super Duper 750ml,16,0.75,l,parsed_product_name
601,CrissCross Deep Clean Anti-Plaque Toothbrush,4 pcs,4.00,unit,parsed_quantity
632,Paper boat lychee 250ml pet,NaN,0.25,l,parsed_product_name
640,Peanut chocolate laddoos,70gms,0.07,kg,parsed_quantity


In [5]:
# the parser itself, on a few awkward strings
tests = ["500 g", "2 x 200ml", "70gms", "1 Dozen", "1.2 L", "4 pcs", "78.9"]
pd.DataFrame([(t, *N.parse_quantity(t)[:2]) for t in tests], columns=["text", "quantity", "unit"])

,text,quantity,unit
0,500 g,0.50,kg
1,2 x 200ml,0.40,l
2,70gms,0.07,kg
3,1 Dozen,12.00,unit
4,1.2 L,1.20,l
5,4 pcs,4.00,unit
6,78.9,NaN,NaN


## 2 · Brands

Open Food Facts lets anyone type the brand, so the same company shows up in several spellings. I take the first brand listed and pass it through an alias table. Every raw spelling is stored in `dim_brand_alias`, so the mapping can always be audited.

In [6]:
brands = op[op.brand_rule.isin(["alias_mapped", "case_normalised"])][["brand_raw", "brand_std", "brand_rule"]]
brands.drop_duplicates().sort_values("brand_std").head(20)

,brand_raw,brand_std,brand_rule
511,AMUL,Amul,alias_mapped
619,ASUS,Asus,case_normalised
335,Balaji,Balaji Wafers,alias_mapped
604,"BAT, Bait Al Tamur",Bat,case_normalised
453,Brotannica,Britannia,alias_mapped
323,cadbury,Cadbury,case_normalised
477,Coke,Coca-Cola,alias_mapped
484,Coca Cola,Coca-Cola,alias_mapped
627,"Coca-Cola Company, Maaza",Coca-Cola,alias_mapped
575,Dmart Premia,DMart (private label),alias_mapped


## 3 · Categories

The project needs four groups — **Groceries, Packaged Food, Household, Personal Care**. For Open Prices I map the Open Food Facts category tags with ordered rules (most specific first). Electronics and alcohol that people logged in the same app go to *Out of scope*; rows with no tags at all go to *Unclassified*. DoCA commodities use a fixed lookup table.

In [7]:
(df.groupby(["source_code", "category_group", "category"]).size()
   .rename("records").reset_index()
   .sort_values(["source_code", "records"], ascending=[True, False]))

,source_code,category_group,category,records
0,DOCA_PMD,Groceries,Cereals & Flours,58
1,DOCA_PMD,Groceries,Edible Oils & Ghee,48
4,DOCA_PMD,Groceries,Pulses,47
6,DOCA_PMD,Groceries,"Tea, Salt & Spices",44
7,DOCA_PMD,Groceries,Vegetables,39
3,DOCA_PMD,Groceries,"Milk, Dairy & Eggs",19
5,DOCA_PMD,Groceries,Sugar & Gur,14
2,DOCA_PMD,Groceries,Fruits,6
15,OPEN_PRICES,Packaged Food,"Biscuits, Bread & Bakery",55
14,OPEN_PRICES,Packaged Food,Beverages,39


## 4 · Locations

Shops in Open Prices are OpenStreetMap places. I use the city field when present, otherwise I read the city out of the full address, then clean up common variants (*Bangalore → Bengaluru*, *Mumbai Suburban → Mumbai*). State → region comes from a small lookup. DoCA prices are All-India averages, so their location is simply "All India (average)".

In [8]:
op_in = op[op.country_code == "IN"]
print(op_in["city_source"].value_counts().to_string(), "\n")
changed = op_in[op_in.city_raw.notna() & (op_in.city_raw != op_in.city)][["city_raw", "city", "state", "region"]]
changed.drop_duplicates()

city_source
osm_address_city       344
parsed_display_name     40
missing                  5
state_is_city            1 



,city_raw,city,state,region
283,Mumbai City,Mumbai,Maharashtra,West
291,Pimpri-Chinchwad,Pune (Pimpri-Chinchwad),Maharashtra,West
296,New Delhi,Delhi,Delhi,North
299,Bengaluru Urban,Bengaluru,Karnataka,South
311,Ernakulam,Kochi (Ernakulam),Kerala,South
342,Pune City,Pune,Maharashtra,West
354,Kochi,Kochi (Ernakulam),Kerala,South
403,Nagpur City,Nagpur,Maharashtra,West
447,West Delhi,Delhi,Delhi,North
463,Mumbai Suburban,Mumbai,Maharashtra,West


## 5 · Dates and the two kinds of date

Every price has an **observation date** (when that price applied) and a **collection date** (when my pipeline captured it). For DoCA these differ a lot: the 2-years-back value is *observed* on 30 Sep 2024 but *collected* on 1 Oct 2026.

In [9]:
df.groupby("source_code").agg(first_observation=("observation_date", "min"),
                              last_observation=("observation_date", "max"),
                              collected_on=("collection_date", "max"))

,first_observation,last_observation,collected_on
source_code,,,
DOCA_PMD,2024-09-30,2026-09-30,2026-10-01
OPEN_PRICES,2023-11-23,2026-09-29,2026-10-01


## 6 · Why I never compare pack prices directly

Same brand, very different packs — the pack price ranks them the wrong way round:

In [10]:
demo = op[(op.brand_std == "Amul") & op.qty_std.notna()].copy()
demo["price_per_unit"] = (demo["price"] / demo["qty_std"]).round(1)
demo[["product_name_std", "qty_std", "std_unit", "price", "price_per_unit"]].sort_values("price")

,product_name_std,qty_std,std_unit,price,price_per_unit
512,T-Special Premium Dairy Whitener,0.022,kg,10.0,454.5
577,Amul Spiced Buttermilk,0.200,l,20.0,100.0
642,Kashmiri Kulfi,0.048,kg,20.0,416.7
565,Amul kool elaichi 180ml,0.180,l,25.0,138.9
503,Berry Dazzle,0.125,l,40.0,320.0
671,Badam Shakers,0.200,l,40.0,200.0
633,ENERDOZE power up,0.250,l,60.0,240.0
637,Buffalo milk A2 milk,1.000,l,90.0,90.0
544,Malai Paneer,0.200,kg,95.0,475.0
547,Jiralu Masala Ghee Khakhra,0.200,kg,100.0,500.0


## Takeaways

- All prices now have a comparable **₹/kg, ₹/litre or ₹/unit** figure (where the pack size is known).
- Brands, categories, cities and dates follow written rules that can be checked line by line.
- Nothing was deleted. Problems found here become **flags** in notebook 04.

Next: **03 · Building the database**.